# 5. Value of flow temperature data

Part 5 of 5. How much better could weak heat pumps be identified if measured flow temperature were available alongside design and electricity data?

Notebook 4 showed that electricity mostly reflects heat demand rather than efficiency. Physically, heat pump efficiency depends most on the gap between flow and outdoor temperature, so flow temperature is the obvious missing piece.

Flow temperature comes from heating system sensors, not from an electricity meter. This is a value-of-information test, not part of the main model: it shows what such data would be worth if it were available, for example from heat pump controllers or manufacturer platforms.

Same main model and testing scheme as notebook 3. Runtime is about 2 minutes with Ridge.

## Step 1: Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/EoH_project'      # the raw EoH files
OUT_DIR = DATA_DIR + '/q1_paper'                     # everything this paper series makes
FIG_DIR = OUT_DIR + '/figures'
TABLE_DIR = OUT_DIR + '/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2f6f9f'
ORANGE = '#c96a14'
GREY = '#8a8f98'
GREEN = '#3f8f5a'
MAKER = {'Daikin': 'Manufacturer A', 'Mitsubishi': 'Manufacturer B', 'Vaillant': 'Manufacturer C'}
MAKER_COLOR = {'Daikin': '#2f6f9f', 'Mitsubishi': '#8a5aa8', 'Vaillant': '#5b8c5a'}

B = 2000
REPEATS = 10
inner_cv = KFold(n_splits=5, shuffle=True, random_state=0)


def make_ridge():
    return make_pipeline(StandardScaler(), RidgeCV(alphas=[0.1, 1, 3, 10, 30, 100, 300]))


def make_forest():
    forest = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=1)
    settings = {'min_samples_leaf': [5, 10, 20], 'max_features': [0.33, 1.0]}
    return GridSearchCV(forest, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


def make_lightgbm():
    booster = LGBMRegressor(n_estimators=300, learning_rate=0.03, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, random_state=0, n_jobs=1, verbose=-1)
    settings = {'num_leaves': [3, 7], 'min_child_samples': [10, 20, 40]}
    return GridSearchCV(booster, settings, cv=inner_cv, scoring='neg_mean_squared_error', n_jobs=-1)


makers = {'Ridge': make_ridge, 'Random forest': make_forest, 'LightGBM': make_lightgbm}
print('ready')

## Step 2: Flow temperature inputs

Four inputs from the summary file, averaged over each home's 12-month window:
- `Mean_annual_SH_flow_temp_selected_window`: mean space-heating flow temperature
- `Winter_mean_SH_flow_temp_selected_window`: the same, winter only
- `Mean_annual_HW_flow_temp_selected_window`: mean flow temperature during hot water production
- `flow_minus_outside`: space-heating flow temperature minus mean outdoor temperature

The table shows coverage, median and rank correlation with SPF for each input, with annual electricity for comparison.

In [ ]:
data = pd.read_csv(OUT_DIR + '/q1_table.csv')
with open(OUT_DIR + '/q1_input_sets.json') as f:
    input_sets = json.load(f)
with open(OUT_DIR + '/q1_main_model.json') as f:
    MAIN_MODEL = json.load(f)['main_model']

data['flow_minus_outside'] = data['Mean_annual_SH_flow_temp_selected_window'] - data['mean_outside_temp']
flow_inputs = ['Mean_annual_SH_flow_temp_selected_window', 'Winter_mean_SH_flow_temp_selected_window',
               'Mean_annual_HW_flow_temp_selected_window', 'flow_minus_outside']
spf = data['SPF'].values
weak = spf < 2.5
base_rate = 100 * weak.mean()


def spearman(a, b):
    ok = ~(pd.isna(a) | pd.isna(b))
    return np.corrcoef(rankdata(np.asarray(a)[ok]), rankdata(np.asarray(b)[ok]))[0, 1]


print('homes:', len(data), '| main model:', MAIN_MODEL)
rows = {}
for c in flow_inputs + ['annual_kWh']:
    rows[c] = {'homes filled': data[c].notna().sum(), 'median': data[c].median(), 'rank corr with SPF': spearman(data[c], data['SPF'])}
pd.DataFrame(rows).T.round(3)

## Step 3: A first look

Each point is a home, coloured by manufacturer, with the weak threshold as a red dashed line. Left: mean space-heating flow temperature. Right: flow minus outdoor temperature.

In [ ]:
fig = plt.figure(figsize=(10, 4))
for k, (col, label) in enumerate([('Mean_annual_SH_flow_temp_selected_window', 'Mean space-heating flow temperature (C)'),
                                  ('flow_minus_outside', 'Flow temperature minus outdoor temperature (K)')]):
    plt.subplot(1, 2, k + 1)
    for brand in ['Daikin', 'Mitsubishi', 'Vaillant']:
        part = data[data['HP_Brand'] == brand]
        plt.scatter(part[col], part['SPF'], s=8, alpha=0.6, color=MAKER_COLOR[brand], label=MAKER[brand])
    plt.axhline(2.5, color='red', linestyle=':', linewidth=1)
    plt.xlabel(label)
    plt.ylabel('Measured SPF')
    plt.title(f"({'ab'[k]}) rank corr = {spearman(data[col], data['SPF']):.2f}")
    if k == 0:
        plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_05_step03_flow_vs_spf.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_05_step03_flow_vs_spf.png')

## Step 4: Helper functions

Same as notebooks 3 and 4: unseen-home predictions from 5 folds and 10 repeats, the same metrics, and paired bootstrap intervals.

In [ ]:
def precision_flagged(pred, weak_part, share=0.25):
    n_flag = int(round(share * len(pred)))
    flagged = np.argsort(pred, kind='stable')[:n_flag]
    return 100 * weak_part[flagged].mean()


def scores(pred_matrix, idx=None):
    if idx is None:
        idx = np.arange(len(spf))
    spf_rank = rankdata(spf[idx])
    precisions = []
    ranks = []
    for r in range(pred_matrix.shape[1]):
        p = pred_matrix[idx, r]
        precisions.append(precision_flagged(p, weak[idx]))
        ranks.append(np.corrcoef(spf_rank, rankdata(p))[0, 1])
    return np.mean(precisions), np.mean(ranks)


def ci(values):
    return np.percentile(values, 2.5), np.percentile(values, 97.5)


boot_idx = np.random.default_rng(1).integers(0, len(spf), size=(B, len(spf)))


def cv_predictions(inputs):
    pred = np.zeros((len(data), REPEATS))
    for r in range(REPEATS):
        for train_idx, test_idx in KFold(n_splits=5, shuffle=True, random_state=r).split(data):
            train = data.iloc[train_idx]
            test = data.iloc[test_idx]
            medians = train[inputs].median()
            model = makers[MAIN_MODEL]()
            model.fit(train[inputs].fillna(medians).fillna(0), train['SPF'])
            pred[test_idx, r] = model.predict(test[inputs].fillna(medians).fillna(0))
    return pred


print('tools ready')

## Step 5: Main table

Six input sets with the main model: electricity only, flow temperatures only, design only, design + flow temperatures, the main model (design + electricity), and the main model + flow temperatures.

Columns: flagged precision, ratio to random, rank correlation and test R2. The design + electricity row should match notebook 3.

In [ ]:
methods = {'electricity only': input_sets['electricity'],
           'flow temperatures only': flow_inputs,
           'design only': input_sets['design'],
           'design + flow temperatures': input_sets['design'] + flow_inputs,
           'design + electricity (main)': input_sets['both'],
           'design + electricity + flow temperatures': input_sets['both'] + flow_inputs}

predictions = {}
boot = {}
rows = []
for name, inputs in methods.items():
    pred = cv_predictions(inputs)
    predictions[name] = pred
    point = scores(pred)
    boot[name] = np.array([scores(pred, i) for i in boot_idx])
    rows.append({'method': name, 'precision (%)': point[0], 'low': ci(boot[name][:, 0])[0], 'high': ci(boot[name][:, 0])[1],
                 'times random': point[0] / base_rate, 'rank corr': point[1],
                 'rank low': ci(boot[name][:, 1])[0], 'rank high': ci(boot[name][:, 1])[1],
                 'R2 test': np.mean([r2_score(spf, pred[:, r]) for r in range(REPEATS)])})
    print(name, ': done')
flow_table = pd.DataFrame(rows).set_index('method')
print(f'random flagging: {base_rate:.1f}%')
flow_table.round(3)

## Step 6: Paired comparisons

Differences on the same homes with 95% intervals, and the share of resamples in which the first method wins:
1. flow temperatures added to the main model
2. flow temperatures added to design data
3. four flow temperatures against a full year of electricity

In [ ]:
pairs = [('1. flow temperatures added to the main model', 'design + electricity + flow temperatures', 'design + electricity (main)'),
         ('2. flow temperatures added to design data', 'design + flow temperatures', 'design only'),
         ('3. flow temperatures only vs electricity only', 'flow temperatures only', 'electricity only')]
rows = []
for label, a, c in pairs:
    diff = boot[a] - boot[c]
    point = np.array(scores(predictions[a])) - np.array(scores(predictions[c]))
    rows.append({'comparison': label,
                 'precision change (points)': point[0], 'low': ci(diff[:, 0])[0], 'high': ci(diff[:, 0])[1],
                 'better (% of resamples)': 100 * (diff[:, 0] > 0).mean(),
                 'rank corr change': point[1], 'rank low': ci(diff[:, 1])[0], 'rank high': ci(diff[:, 1])[1],
                 'rank better (% of resamples)': 100 * (diff[:, 1] > 0).mean()})
flow_pairs = pd.DataFrame(rows).set_index('comparison')
flow_pairs.round(3)

## Step 7: Value of information figure

Blue: data available without extra sensors (electricity, design). Green: with measured flow temperature.

(a) Flagged precision; the dashed line is random flagging.
(b) Rank correlation with measured SPF.

In [ ]:
show = [('electricity only', 'Electricity only', BLUE),
        ('design only', 'Design data only', BLUE),
        ('design + electricity (main)', 'Design + electricity (main model)', BLUE),
        ('flow temperatures only', 'Measured flow temperatures only', GREEN),
        ('design + flow temperatures', 'Design + flow temperatures', GREEN),
        ('design + electricity + flow temperatures', 'Main model + flow temperatures', GREEN)]
part = flow_table.loc[[s[0] for s in show]]
names = [s[1] for s in show]
colors = [s[2] for s in show]

fig = plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.barh(names, part['precision (%)'], xerr=[part['precision (%)'] - part['low'], part['high'] - part['precision (%)']],
         capsize=3, color=colors)
plt.axvline(base_rate, color='black', linestyle='--', linewidth=1, label=f'Random flagging ({base_rate:.1f}%)')
plt.gca().invert_yaxis()
plt.xlim(0, 90)
plt.xlabel('Flagged homes with a weak heat pump (%)')
plt.title('(a) Flagging the lowest 25%')
plt.legend(loc='lower right')
plt.subplot(1, 2, 2)
plt.barh(names, part['rank corr'], xerr=[part['rank corr'] - part['rank low'], part['rank high'] - part['rank corr']],
         capsize=3, color=colors)
plt.gca().invert_yaxis()
plt.gca().set_yticklabels([])
plt.xlim(0, 0.8)
plt.xlabel('Rank correlation with measured SPF')
plt.title('(b) Ranking all homes')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_05_step07_value_of_information.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_05_step07_value_of_information.png')

## Step 8: Save tables

In [ ]:
flow_table.to_csv(TABLE_DIR + '/q1_05_flow_temperature.csv')
flow_pairs.to_csv(TABLE_DIR + '/q1_05_flow_temperature_pairs.csv')
print('saved 2 tables in', TABLE_DIR)